# Import Pyspark Libraries and Create Spark Session

In [1]:
from pyspark.sql import SparkSession,functions as F
from pyspark.sql.types import StructType, StructField,StringType,IntegerType,FloatType

#THESE ARE USED FOR DATA MANIPULATION

In [2]:
spark=SparkSession.builder.appName('Data Analysis').master('local[*]').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/24 11:54:33 WARN Utils: Your hostname, Samriddhas-Macbook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.1.9.20 instead (on interface en0)
26/05/24 11:54:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/24 11:54:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/24 11:54:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


# Importing Data

In [3]:
employees_data=spark.read.csv('employees-1103c135-913c-424a-8696-e304e17ffee4.csv',header=True,inferSchema=True)


#Kaile kai chai esari manually data type dina parcha in the form of schema when inferSchema=True dida ramrari data types diyeko hunna
'''
schema=StructType(
    [
        StructField('id',IntegerType(),nullable=False)
    ]
)
'''

#json read garna=spark.read.json()
#parquet read garna=spark.read.parquet()

"\nschema=StructType(\n    [\n        StructField('id',IntegerType(),nullable=False)\n    ]\n)\n"

In [4]:
print(employees_data) #does not show output

employees_data.show()

DataFrame[id: int, name: string, department: string, salary: int, age: int]
+---+-----+-----------+------+----+
| id| name| department|salary| age|
+---+-----+-----------+------+----+
|  1|Alice|Engineering| 85000|  29|
|  2|  Bob|  Marketing| 62000|  34|
|  3|Carol|Engineering| 91000|  41|
|  4|David|         HR|  NULL|  27|
|  5|  Eve|  Marketing| 70000|  38|
|  6|Frank|       NULL| 54000|NULL|
|  7|Grace|Engineering| 95000|  35|
|  8|Henry|         HR| 58000|  45|
|  9|  Ivy|  Marketing| 67000|  31|
| 10| Jack|Engineering| 78000|  28|
+---+-----+-----------+------+----+



In [5]:
employees_data.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- age: integer (nullable = true)



In [6]:
employees_data.dtypes

[('id', 'int'),
 ('name', 'string'),
 ('department', 'string'),
 ('salary', 'int'),
 ('age', 'int')]

In [7]:
#check for any null values

from pyspark.sql.functions import col,sum as spark_sum,isnan,when

In [8]:
null_counts=employees_data.select([
    spark_sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
    for c in employees_data.columns
])

In [9]:
null_counts.show()

+---+----+----------+------+---+
| id|name|department|salary|age|
+---+----+----------+------+---+
|  0|   0|         1|     1|  1|
+---+----+----------+------+---+



In [10]:
employees_data.filter(col('salary').isNull()).show()

+---+-----+----------+------+---+
| id| name|department|salary|age|
+---+-----+----------+------+---+
|  4|David|        HR|  NULL| 27|
+---+-----+----------+------+---+



In [11]:
employees_data.filter(col('name').isNull()).show()

+---+----+----------+------+---+
| id|name|department|salary|age|
+---+----+----------+------+---+
+---+----+----------+------+---+



In [12]:
employees_data_cleaned=employees_data.dropna()

employees_data_cleaned.show()

+---+-----+-----------+------+---+
| id| name| department|salary|age|
+---+-----+-----------+------+---+
|  1|Alice|Engineering| 85000| 29|
|  2|  Bob|  Marketing| 62000| 34|
|  3|Carol|Engineering| 91000| 41|
|  5|  Eve|  Marketing| 70000| 38|
|  7|Grace|Engineering| 95000| 35|
|  8|Henry|         HR| 58000| 45|
|  9|  Ivy|  Marketing| 67000| 31|
| 10| Jack|Engineering| 78000| 28|
+---+-----+-----------+------+---+



In [13]:
employees_data_filled=employees_data.fillna({'salary':0,'department':'Unknown'})

employees_data_filled.show()

+---+-----+-----------+------+----+
| id| name| department|salary| age|
+---+-----+-----------+------+----+
|  1|Alice|Engineering| 85000|  29|
|  2|  Bob|  Marketing| 62000|  34|
|  3|Carol|Engineering| 91000|  41|
|  4|David|         HR|     0|  27|
|  5|  Eve|  Marketing| 70000|  38|
|  6|Frank|    Unknown| 54000|NULL|
|  7|Grace|Engineering| 95000|  35|
|  8|Henry|         HR| 58000|  45|
|  9|  Ivy|  Marketing| 67000|  31|
| 10| Jack|Engineering| 78000|  28|
+---+-----+-----------+------+----+



In [14]:
employees_data.describe().show()

26/05/24 11:54:41 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+-----+-----------+-----------------+-----------------+
|summary|                id| name| department|           salary|              age|
+-------+------------------+-----+-----------+-----------------+-----------------+
|  count|                10|   10|          9|                9|                9|
|   mean|               5.5| NULL|       NULL|73333.33333333333|34.22222222222222|
| stddev|3.0276503540974917| NULL|       NULL|14696.93845669907|6.180165405913052|
|    min|                 1|Alice|Engineering|            54000|               27|
|    max|                10| Jack|  Marketing|            95000|               45|
+-------+------------------+-----+-----------+-----------------+-----------------+



In [15]:
employees_data.describe('salary').show()

+-------+-----------------+
|summary|           salary|
+-------+-----------------+
|  count|                9|
|   mean|73333.33333333333|
| stddev|14696.93845669907|
|    min|            54000|
|    max|            95000|
+-------+-----------------+



# Selecting Columns

In [16]:
employees_data.select('name').show()

+-----+
| name|
+-----+
|Alice|
|  Bob|
|Carol|
|David|
|  Eve|
|Frank|
|Grace|
|Henry|
|  Ivy|
| Jack|
+-----+



In [17]:
employees_data.select(col('name'),col('salary'),col('department')).show()

+-----+------+-----------+
| name|salary| department|
+-----+------+-----------+
|Alice| 85000|Engineering|
|  Bob| 62000|  Marketing|
|Carol| 91000|Engineering|
|David|  NULL|         HR|
|  Eve| 70000|  Marketing|
|Frank| 54000|       NULL|
|Grace| 95000|Engineering|
|Henry| 58000|         HR|
|  Ivy| 67000|  Marketing|
| Jack| 78000|Engineering|
+-----+------+-----------+



In [18]:
employees_data.select(
    col('name'),
    col('salary'),
    (col('salary') * 2).alias('salary_with_bonus')
).show()

+-----+------+-----------------+
| name|salary|salary_with_bonus|
+-----+------+-----------------+
|Alice| 85000|           170000|
|  Bob| 62000|           124000|
|Carol| 91000|           182000|
|David|  NULL|             NULL|
|  Eve| 70000|           140000|
|Frank| 54000|           108000|
|Grace| 95000|           190000|
|Henry| 58000|           116000|
|  Ivy| 67000|           134000|
| Jack| 78000|           156000|
+-----+------+-----------------+



In [19]:
employees_data=employees_data.withColumn('salary_band',when(col('salary')>=80000,'High')
                                        .when(col('salary')>=60000,'Mid')
                                        .otherwise("Low")
                                        )

employees_data.show()

+---+-----+-----------+------+----+-----------+
| id| name| department|salary| age|salary_band|
+---+-----+-----------+------+----+-----------+
|  1|Alice|Engineering| 85000|  29|       High|
|  2|  Bob|  Marketing| 62000|  34|        Mid|
|  3|Carol|Engineering| 91000|  41|       High|
|  4|David|         HR|  NULL|  27|        Low|
|  5|  Eve|  Marketing| 70000|  38|        Mid|
|  6|Frank|       NULL| 54000|NULL|        Low|
|  7|Grace|Engineering| 95000|  35|       High|
|  8|Henry|         HR| 58000|  45|        Low|
|  9|  Ivy|  Marketing| 67000|  31|        Mid|
| 10| Jack|Engineering| 78000|  28|        Mid|
+---+-----+-----------+------+----+-----------+



In [22]:
# employees_data=employees_data.select(
#     (col('salary') * 2).alias('salary_with_bonus')
# )

# Rows selection

In [30]:
employees_data.filter(col('salary')>65000).show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  3|Carol|Engineering| 91000| 41|       High|
|  5|  Eve|  Marketing| 70000| 38|        Mid|
|  7|Grace|Engineering| 95000| 35|       High|
|  9|  Ivy|  Marketing| 67000| 31|        Mid|
| 10| Jack|Engineering| 78000| 28|        Mid|
+---+-----+-----------+------+---+-----------+



In [31]:
# Department engineering

employees_data.where(col('department')=="Engineering").show()

#Spark ma que aayo re question kun kun method bata row select garna milcha bhanyo bhane we need to use filter or where

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  3|Carol|Engineering| 91000| 41|       High|
|  7|Grace|Engineering| 95000| 35|       High|
| 10| Jack|Engineering| 78000| 28|        Mid|
+---+-----+-----------+------+---+-----------+



In [37]:
employees_data.where((col('age')<40) & (col('salary')>60000)).show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  2|  Bob|  Marketing| 62000| 34|        Mid|
|  5|  Eve|  Marketing| 70000| 38|        Mid|
|  7|Grace|Engineering| 95000| 35|       High|
|  9|  Ivy|  Marketing| 67000| 31|        Mid|
| 10| Jack|Engineering| 78000| 28|        Mid|
+---+-----+-----------+------+---+-----------+



In [40]:
employees_data.limit(3).show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  2|  Bob|  Marketing| 62000| 34|        Mid|
|  3|Carol|Engineering| 91000| 41|       High|
+---+-----+-----------+------+---+-----------+



In [41]:
employees_data_copy=employees_data

In [42]:
employees_data_copy.show()

+---+-----+-----------+------+----+-----------+
| id| name| department|salary| age|salary_band|
+---+-----+-----------+------+----+-----------+
|  1|Alice|Engineering| 85000|  29|       High|
|  2|  Bob|  Marketing| 62000|  34|        Mid|
|  3|Carol|Engineering| 91000|  41|       High|
|  4|David|         HR|  NULL|  27|        Low|
|  5|  Eve|  Marketing| 70000|  38|        Mid|
|  6|Frank|       NULL| 54000|NULL|        Low|
|  7|Grace|Engineering| 95000|  35|       High|
|  8|Henry|         HR| 58000|  45|        Low|
|  9|  Ivy|  Marketing| 67000|  31|        Mid|
| 10| Jack|Engineering| 78000|  28|        Mid|
+---+-----+-----------+------+----+-----------+



In [45]:
employees_data_copy.drop('age').show() #for column deletion

+---+-----+-----------+------+-----------+
| id| name| department|salary|salary_band|
+---+-----+-----------+------+-----------+
|  1|Alice|Engineering| 85000|       High|
|  2|  Bob|  Marketing| 62000|        Mid|
|  3|Carol|Engineering| 91000|       High|
|  4|David|         HR|  NULL|        Low|
|  5|  Eve|  Marketing| 70000|        Mid|
|  6|Frank|       NULL| 54000|        Low|
|  7|Grace|Engineering| 95000|       High|
|  8|Henry|         HR| 58000|        Low|
|  9|  Ivy|  Marketing| 67000|        Mid|
| 10| Jack|Engineering| 78000|        Mid|
+---+-----+-----------+------+-----------+



In [47]:
employees_data_copy.filter(col('department')!='HR').show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  2|  Bob|  Marketing| 62000| 34|        Mid|
|  3|Carol|Engineering| 91000| 41|       High|
|  5|  Eve|  Marketing| 70000| 38|        Mid|
|  7|Grace|Engineering| 95000| 35|       High|
|  9|  Ivy|  Marketing| 67000| 31|        Mid|
| 10| Jack|Engineering| 78000| 28|        Mid|
+---+-----+-----------+------+---+-----------+



In [50]:
employees_data_copy.filter((col('department') != 'HR') & (col('department') != 'Marketing')).show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  3|Carol|Engineering| 91000| 41|       High|
|  7|Grace|Engineering| 95000| 35|       High|
| 10| Jack|Engineering| 78000| 28|        Mid|
+---+-----+-----------+------+---+-----------+



In [52]:
employees_data_copy.withColumnRenamed('salary','annual_salary').show()

+---+-----+-----------+-------------+----+-----------+
| id| name| department|annual_salary| age|salary_band|
+---+-----+-----------+-------------+----+-----------+
|  1|Alice|Engineering|        85000|  29|       High|
|  2|  Bob|  Marketing|        62000|  34|        Mid|
|  3|Carol|Engineering|        91000|  41|       High|
|  4|David|         HR|         NULL|  27|        Low|
|  5|  Eve|  Marketing|        70000|  38|        Mid|
|  6|Frank|       NULL|        54000|NULL|        Low|
|  7|Grace|Engineering|        95000|  35|       High|
|  8|Henry|         HR|        58000|  45|        Low|
|  9|  Ivy|  Marketing|        67000|  31|        Mid|
| 10| Jack|Engineering|        78000|  28|        Mid|
+---+-----+-----------+-------------+----+-----------+



In [60]:
# Change the column names of entire table

new_names=['emp_id','full_name','dept','pay','years','s_band']

employees_data_copy_2=employees_data_copy.toDF(*new_names)

In [61]:
employees_data_copy_2.show()

+------+---------+-----------+-----+-----+------+
|emp_id|full_name|       dept|  pay|years|s_band|
+------+---------+-----------+-----+-----+------+
|     1|    Alice|Engineering|85000|   29|  High|
|     2|      Bob|  Marketing|62000|   34|   Mid|
|     3|    Carol|Engineering|91000|   41|  High|
|     4|    David|         HR| NULL|   27|   Low|
|     5|      Eve|  Marketing|70000|   38|   Mid|
|     6|    Frank|       NULL|54000| NULL|   Low|
|     7|    Grace|Engineering|95000|   35|  High|
|     8|    Henry|         HR|58000|   45|   Low|
|     9|      Ivy|  Marketing|67000|   31|   Mid|
|    10|     Jack|Engineering|78000|   28|   Mid|
+------+---------+-----------+-----+-----+------+



In [64]:
employees_data_copy.orderBy(col('salary').desc()).show()

+---+-----+-----------+------+----+-----------+
| id| name| department|salary| age|salary_band|
+---+-----+-----------+------+----+-----------+
|  7|Grace|Engineering| 95000|  35|       High|
|  3|Carol|Engineering| 91000|  41|       High|
|  1|Alice|Engineering| 85000|  29|       High|
| 10| Jack|Engineering| 78000|  28|        Mid|
|  5|  Eve|  Marketing| 70000|  38|        Mid|
|  9|  Ivy|  Marketing| 67000|  31|        Mid|
|  2|  Bob|  Marketing| 62000|  34|        Mid|
|  8|Henry|         HR| 58000|  45|        Low|
|  6|Frank|       NULL| 54000|NULL|        Low|
|  4|David|         HR|  NULL|  27|        Low|
+---+-----+-----------+------+----+-----------+



In [66]:
employees_data_copy.groupby('department').agg(F.mean('salary').alias('avg_salary_count')).show()

+-----------+-----------------+
| department| avg_salary_count|
+-----------+-----------------+
|Engineering|          87250.0|
|         HR|          58000.0|
|       NULL|          54000.0|
|  Marketing|66333.33333333333|
+-----------+-----------------+



In [67]:
employees_data_copy_2=employees_data_copy.dropna()

In [69]:
employees_data_copy_2.show()

+---+-----+-----------+------+---+-----------+
| id| name| department|salary|age|salary_band|
+---+-----+-----------+------+---+-----------+
|  1|Alice|Engineering| 85000| 29|       High|
|  2|  Bob|  Marketing| 62000| 34|        Mid|
|  3|Carol|Engineering| 91000| 41|       High|
|  5|  Eve|  Marketing| 70000| 38|        Mid|
|  7|Grace|Engineering| 95000| 35|       High|
|  8|Henry|         HR| 58000| 45|        Low|
|  9|  Ivy|  Marketing| 67000| 31|        Mid|
| 10| Jack|Engineering| 78000| 28|        Mid|
+---+-----+-----------+------+---+-----------+



In [71]:
employees_data_copy_2.write.csv('output/employees_clean',header=True,mode='overwrite')

In [73]:
spark.stop()